# Kantar Synthetic Survey System - Quick Start

This notebook provides a quick introduction to the Kantar synthetic survey system.

## What This System Does

The system generates synthetic survey respondents using **Semantic Similarity Rating (SSR)** methodology:
1. Creates realistic personas with demographics
2. Generates natural language responses using LLMs
3. Maps responses to rating scales using semantic similarity
4. Outputs data in exact Kantar Excel format
5. Validates against ground truth

## Two Generation Modes

1. **Ground Truth Mode**: Generate data for existing studies (requires PPTX + Excel files)
2. **No Ground Truth Mode**: Test new concepts without historical data (uses market profiles)

## Prerequisites

- OpenAI API key set in environment
- For ground truth mode: Study data in `data/kantar-survey-source/`

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Setup paths
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Load environment
load_dotenv(project_root / '.env')

# Check API key
if not os.getenv('OPENAI_API_KEY'):
    print("⚠️  OPENAI_API_KEY not found in environment")
    print("   Set it with: export OPENAI_API_KEY='your-key'")
else:
    print("✓ OpenAI API key found")

print(f"✓ Project root: {project_root}")

## Explore Available Studies

In [ ]:
from src.kantar.study_catalog import StudyCatalog

# Initialize catalog
catalog = StudyCatalog()

print("Available Studies:")
print("=" * 80)

for study in catalog.studies.values():
    print(f"\n📁 {study.study_id}: {study.study_name}")
    print(f"   Markets: {', '.join(study.complete_markets)}")
    print(f"   Concepts: {len(study.markets[study.complete_markets[0]].concepts) if study.complete_markets else 'N/A'}")
    print(f"   Status: {'✓ Complete' if study.complete_markets else '⚠️ Incomplete'}")

## View Available Market Profiles

For no-ground-truth mode, you can use pre-built market profiles:

In [ ]:
from src.kantar.market_profiles import list_market_profiles

print("Available Market Profiles:")
print("=" * 80)

for profile_id, info in list_market_profiles().items():
    print(f"\n🌍 {profile_id}")
    print(f"   Name: {info['name']}")
    print(f"   Description: {info['description']}")

## Quick Example: Generate 5 Respondents

Let's generate a small test dataset using ground truth mode:

In [ ]:
from src.kantar.survey_runner import KantarSurveyRunner

# Initialize runner
runner = KantarSurveyRunner(model="gpt-4o-mini")

# Find first study with complete markets
study = None
for s in catalog.studies.values():
    if s.complete_markets:
        study = s
        break

if not study:
    print("⚠️  No complete studies found. Please add study data to data/kantar-survey-source/")
else:
    market = study.complete_markets[0]
    
    print(f"Generating 5 respondents for {study.study_id} - {market}...")
    print("This will take 1-2 minutes...\n")
    
    output_file = runner.generate_for_market(
        study_id=study.study_id,
        market_code=market,
        num_respondents=5,
        use_ground_truth_demographics=False
    )
    
    print(f"\n✓ Generated: {output_file}")

## Load and Preview Generated Data

In [ ]:
import pandas as pd

# Load the generated file
df = pd.read_excel(output_file)

print(f"Generated Data: {len(df)} rows × {len(df.columns)} columns\n")

# Show demographics
print("Sample Demographics:")
demo_cols = ['Gender', 'AGE']
available_demo = [c for c in demo_cols if c in df.columns]
if available_demo:
    print(df[available_demo].head())

# Show sample questions
print("\nSample Questions:")
question_cols = [c for c in df.columns if '(' in c and ')' in c][:3]
if question_cols:
    for col in question_cols:
        print(f"  {col}")
        print(f"    {df[col].value_counts().head(3).to_dict()}\n")

## Next Steps

- **Notebook 02**: Learn how to generate larger datasets with both modes
- **Notebook 03**: Validate and create reports
- **README.md**: Full CLI documentation

## Quick CLI Reference

```bash
# Generate with ground truth
python -m src.kantar.survey_runner \
  --study 61405445-01 \
  --market US \
  --num-respondents 50 \
  --model gpt-4o-mini

# Generate from custom concepts
python -m src.kantar.survey_runner \
  --concepts-file my_concepts.json \
  --num-respondents 100 \
  --market-profile US_gaming \
  --model gpt-4o-mini

# Validate and generate report
python -m src.kantar.validation_runner \
  --study 61405445-01 \
  --market US \
  --synthetic data/synthetic/*.xlsx \
  --generate-report
```